## Actividad 3_20: Perros y gatos
<div style="border-style:groove;border-width:thin;padding:10px">
En esta actividad vamos a utilizar las técnicas de redes neuronales y deep learning que hemos visto en clase para enseñar a este software a diferenciar entre perros y gatos.

Para ello vamos a cargar los datos y etiquetarlos, a lanzar un Random Forest Classifier para establecer un punto de partida que debemos mejorar y después, vamos a tratar de solucionar el problema con una red neuronal convencional.
</div>

In [2]:
from os import listdir
from numpy import asarray
from numpy import save
import tensorflow as tf
#tf.config.set_visible_devices([], 'GPU')
from tensorflow.keras.utils import load_img
from tensorflow.keras.utils import img_to_array

folders = listdir('PetImages')
#Clase Cat será la clase 0.0
#Clase Dog será la clase 1.0

photos =  []
labels = []

2025-05-09 18:34:53.433025: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-05-09 18:34:55.082934: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [3]:
#Importamos las imágenes. Hemos metido un try and catch porque hay dos archivos corruptos que dan error al cargarlos.
#Por razones de memoria cojo solo 5000 imágenes de cada clase.
for idx,folder in enumerate(folders):
    for file in listdir('PetImages/'+folder)[:5000]:
        #Cargamos la imagen.
        #print(len(listdir('PetImages/'+folder)))
        try:    
            photo = load_img('PetImages/'+folder+'/' + file, target_size=(250, 250),color_mode='grayscale')
            #Convertimos la imagen a un array y la guardamos en la lista.
            photos.append(img_to_array(photo))
            #También guardamos la etiqueta.
            labels.append(float(idx))
            #Usamos del para borrar los datos temporales para no ocupar memoria que vamos a necesitar.
            del photo
        except:
            print('Error en la foto ' + file + ' de la carpeta ' + folder)

In [4]:
#Ponemos todas las imágenes como un numpy array de dos dimensiones.
photos_array = asarray(photos)
del photos

In [5]:
print(photos_array.shape)

(10000, 250, 250, 1)


In [6]:
#Aplanamos las fotos para hacer un random forest.
photos_reshape = photos_array.reshape(photos_array.shape[0],-1)
photos_reshape.shape

(10000, 62500)

In [7]:
#Escalamos los datos con StandardScarler. También podríamos dividir los datos entre 255.0
from sklearn.preprocessing import StandardScaler
X = photos_reshape
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [8]:
y = asarray(labels)

In [9]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.1, random_state = 0)

In [10]:
#RANDOM FOREST
#Probamos en primer lugar con RandomForestClassifier para establecer una base de cara a saber si la red neuronal
#está bien o no.
from sklearn.ensemble import RandomForestClassifier
rnd_clf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=0)
rnd_clf.fit(X_train,y_train)
from sklearn.metrics import accuracy_score
y_pred = rnd_clf.predict(X_test)
print(accuracy_score(y_test,y_pred))

0.63


Vemos que el resultado con Random Forest es que el sistema acierta el 63% de las imágenes. Con más imágenes el resultado es un poco mejor, pero no mucho. El resultado no es muy bueno ya que, a priori, un decisor que dijera perro o gato aleatoriamente acertaría el 50% de las veces.

Ahora vamos a solucionar el problema con una red neuronal convencional a ver si conseguimos mejorar al Random Forest:

In [11]:
#RED NEURONAL CONVENCIONAL
#Creamos la red neuronal
from tensorflow import keras
model = keras.models.Sequential()
model.add(keras.layers.InputLayer(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(100,activation='relu',kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(20,activation='relu',kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(1,activation='sigmoid',kernel_initializer='glorot_normal'))

2025-05-09 18:37:14.180784: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-05-09 18:37:14.417181: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-05-09 18:37:14.417601: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

In [12]:
#Usamos Nadam. Se podrían usar otros optimizadores, pero el resultado es similar.
model.compile(loss='binary_crossentropy', optimizer = keras.optimizers.Nadam(learning_rate=0.001, beta_1=0.9, beta_2=0.999), metrics=['accuracy'])
early_stopping_cb = keras.callbacks.EarlyStopping(patience=5,
restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=100000,validation_split = 0.1,callbacks=[early_stopping_cb],batch_size=256)

Epoch 1/100000


I0000 00:00:1746808641.578279    9855 service.cc:145] XLA service 0x7b118000de60 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1746808641.578348    9855 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce RTX 2070 with Max-Q Design, Compute Capability 7.5
2025-05-09 18:37:21.737893: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-05-09 18:37:22.046056: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907


 5/32 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.5352 - loss: 0.9967

I0000 00:00:1746808643.283427    9855 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


32/32 ━━━━━━━━━━━━━━━━━━━━ 8s 127ms/step - accuracy: 0.5555 - loss: 0.8060 - val_accuracy: 0.5700 - val_loss: 1.0167
Epoch 2/100000
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - accuracy: 0.6563 - loss: 0.6176 - val_accuracy: 0.5778 - val_loss: 0.8040
Epoch 3/100000
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - accuracy: 0.6895 - loss: 0.5849 - val_accuracy: 0.5778 - val_loss: 0.7680
Epoch 4/100000
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - accuracy: 0.7402 - loss: 0.5348 - val_accuracy: 0.6056 - val_loss: 0.7203
Epoch 5/100000
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.7701 - loss: 0.4912 - val_accuracy: 0.5911 - val_loss: 0.7296
Epoch 6/100000
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - accuracy: 0.8042 - loss: 0.4372 - val_accuracy: 0.6078 - val_loss: 0.7191
Epoch 7/100000
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.8459 - loss: 0.3796 - val_accuracy: 0.5900 - val_loss: 0.7452
Epoch 8/100000
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.8879 - loss: 0.3108 - val_ac

In [13]:
model.evaluate(X_test,y_test)

32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5534 - loss: 0.7426


[0.7237207293510437, 0.5720000267028809]

Vemos que tiene un 57% de precisión a la hora de clasificar las fotos del conjunto de test. Es mucho peor resultado que con Random Forest. Probemos ahora con una convolucional:

In [ ]:
#RED NEURONAL CONVOLUCIONAL
#Como para la red convolucional necesito los datos en forma de matriz, hago un reshape de los datos que usé antes (aplanados) 
# con la forma que quiero que tengan.
X = photos_reshape.reshape(10000, 250, 250, 1)
X = X/255.0
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.1, random_state = 0)

In [20]:
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPool2D
from tensorflow.keras.layers import Flatten
model = keras.models.Sequential()
model.add(Conv2D(16,(3,3), activation='relu', input_shape = (250,250,1)))
model.add(MaxPool2D(2,2))
model.add(Conv2D(32,(3,3), activation='relu'))
model.add(MaxPool2D(2,2))
model.add(Conv2D(64,(3,3), activation='relu'))
model.add(MaxPool2D(2,2))
model.add(Conv2D(256,(3,3), activation='relu'))
model.add(MaxPool2D(2,2))
model.add(Flatten())
model.add(keras.layers.Dense(100,activation='relu',kernel_initializer='he_normal'))
model.add(keras.layers.Dense(20,activation='relu',kernel_initializer='he_normal'))
model.add(keras.layers.Dense(1,activation='sigmoid',kernel_initializer='glorot_normal'))

In [21]:
#Usamos Nadam. Se podrían usar otros optimizadores, pero el resultado es similar.
model.compile(loss='binary_crossentropy', optimizer = keras.optimizers.Adam(learning_rate=0.0001, beta_1=0.9, beta_2=0.999), metrics=['accuracy'])
early_stopping_cb = keras.callbacks.EarlyStopping(patience=5,
restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=100000,validation_split = 0.1,callbacks=[early_stopping_cb],batch_size=256)

Epoch 1/100000
32/32 ━━━━━━━━━━━━━━━━━━━━ 16s 323ms/step - accuracy: 0.5268 - loss: 0.6936 - val_accuracy: 0.6244 - val_loss: 0.6781
Epoch 2/100000
32/32 ━━━━━━━━━━━━━━━━━━━━ 6s 179ms/step - accuracy: 0.6035 - loss: 0.6713 - val_accuracy: 0.6433 - val_loss: 0.6454
Epoch 3/100000
32/32 ━━━━━━━━━━━━━━━━━━━━ 6s 181ms/step - accuracy: 0.6269 - loss: 0.6486 - val_accuracy: 0.6678 - val_loss: 0.6162
Epoch 4/100000
32/32 ━━━━━━━━━━━━━━━━━━━━ 6s 182ms/step - accuracy: 0.6593 - loss: 0.6201 - val_accuracy: 0.6422 - val_loss: 0.6098
Epoch 5/100000
32/32 ━━━━━━━━━━━━━━━━━━━━ 6s 182ms/step - accuracy: 0.6781 - loss: 0.5995 - val_accuracy: 0.6967 - val_loss: 0.5751
Epoch 6/100000
32/32 ━━━━━━━━━━━━━━━━━━━━ 6s 182ms/step - accuracy: 0.7037 - loss: 0.5740 - val_accuracy: 0.6922 - val_loss: 0.5655
Epoch 7/100000
32/32 ━━━━━━━━━━━━━━━━━━━━ 6s 183ms/step - accuracy: 0.7140 - loss: 0.5600 - val_accuracy: 0.6889 - val_loss: 0.5580
Epoch 8/100000
32/32 ━━━━━━━━━━━━━━━━━━━━ 6s 184ms/step - accuracy: 0.7171 

In [22]:
model.evaluate(X_test,y_test)

32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.7719 - loss: 0.4886


[0.47522735595703125, 0.7760000228881836]